# BRSR Greenwashing Risk Screener — Exploratory Data Analysis

**Project:** Greenwashing Risk Detection in Indian BRSR Disclosures Using NLP  
**Notebook:** Step-by-step EDA walkthrough — shows analytical thinking, not just final results  
**Data:** `outputs/nlp_results.csv` — 15 BRSR filings, FY2022–25, 5 sectors  
**Run after:** `src/nlp_pipeline.py` (generates the CSV this notebook reads)

---

## Table of Contents
1. Setup & Data Loading
2. Corpus Overview
3. Univariate Analysis — Each Metric
4. Bivariate Analysis — SR vs GRS (key relationship)
5. Surprise Finding — AVVR Dominates
6. Sector Comparison
7. Outlier Analysis
8. Readability Analysis
9. Cross-listing Effect
10. Correlation Matrix
11. Statistical Tests
12. Top-5 Actionable Findings
13. Honest Limitations


## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi':130,'font.family':'DejaVu Sans','axes.titlesize':13,'axes.labelsize':11})

# Load pipeline output
df = pd.read_csv('../outputs/nlp_results.csv')
print(f'Loaded {len(df)} companies × {len(df.columns)} columns')
print(f'Sectors: {df["sector"].unique()}')
print(f'FY range: {df["fy"].unique()}')

Loaded 15 companies × 26 columns
Sectors: ['IT' 'Banking' 'Cement' 'Heavy Industry' 'Pharma' 'Auto']
FY range: ['FY2023-24' 'FY2024-25' 'FY2022-23']


## 2. Corpus Overview

In [ ]:
# The 9 core metrics we work with
core_cols = ['company','sector','SR','AVVR','BD','LM_pos','LM_neg','LM_net','FRE','GRS','GRS_tier']
display_df = df[core_cols].sort_values('GRS', ascending=False).reset_index(drop=True)
print(display_df.to_string(index=True))

     company         sector     SR    AVVR      BD  LM_pos  LM_neg  LM_net    FRE   GRS GRS_tier
0   UltraTech         Cement  0.059  0.000  0.0064  0.0149  0.0100  0.0050  23.60  77.8     HIGH
1      Cipla          Pharma  0.059  0.000  0.0000  0.0213  0.0106  0.0106  29.29  77.6     HIGH
2  Tata Steel  Heavy Industry  0.000  0.000  0.0000  0.0099  0.0050  0.0050  24.11  72.5     HIGH
3   Sun Pharma         Pharma  0.000  0.500  0.0000  0.0219  0.0146  0.0073  31.33  67.5     HIGH
4   HDFC Bank        Banking  0.000  0.000  0.0000  0.0123  0.0000  0.0123  28.68  65.0     HIGH
5     Infosys             IT  0.066  0.545  0.0370  0.0194  0.0097  0.0097  24.74  64.0     HIGH
6     Ambuja         Cement   0.000  0.667  0.0000  0.0052  0.0052  0.0000  29.48  56.1   MEDIUM
7        SBI        Banking  0.077  0.333  0.0000  0.0127  0.0000  0.0127  30.45  53.6   MEDIUM
8  Dr. Reddy's       Pharma  0.200  0.750  0.0000  0.0121  0.0182 -0.0061  33.66  53.2   MEDIUM
9         TCS             IT  

## 3. Univariate Analysis

Before plotting, let's look at the summary statistics to understand the distribution of each metric.

**Key things to look for:**
- Is the distribution skewed? (median vs mean)
- How much variation is there? (std dev)
- Are there floor/ceiling effects? (min/max)


In [ ]:
metrics = ['SR','AVVR','BD','LM_pos','LM_neg','GRS','FRE']
desc = df[metrics].describe().round(4)
print(desc.to_string())

print('\n── Interpretation notes ──')
print(f'GRS range: {df["GRS"].min():.1f}–{df["GRS"].max():.1f}  (NO company in LOW band <30)')
print(f'SR median: {df["SR"].median():.3f}  mean: {df["SR"].mean():.3f}  (right-skewed — most companies near-zero)')
print(f'AVVR std: {df["AVVR"].std():.3f}  (high spread — language style varies widely)')
print(f'FRE mean: {df["FRE"].mean():.1f}  (below 30 = "Very Difficult" — BRSR text is hard to read)')
print(f'BD: 9 of 15 companies score exactly 0.0 (most reports have no detectable boilerplate in their extract)')

           SR     AVVR       BD   LM_pos   LM_neg      GRS      FRE
count  15.0000  15.0000  15.0000  15.0000  15.0000  15.0000  15.0000
mean    0.0887   0.4286   0.0063   0.0141   0.0053   57.4467  27.0553
std     0.0815   0.2915   0.0113   0.0077   0.0061   12.8842  11.1715
min     0.0000   0.0000   0.0000   0.0000   0.0000   33.6000   0.2000
25%     0.0294   0.1666   0.0000   0.0106   0.0000   49.1000  24.4250
50%     0.0769   0.5000   0.0000   0.0123   0.0050   53.6000  29.4800
75%     0.1180   0.6667   0.0078   0.0204   0.0099   66.2500  32.9100
max     0.2857   0.8000   0.0370   0.0303   0.0182   77.8000  39.3200

── Interpretation notes ──
GRS range: 33.6–77.8  (NO company in LOW band <30)
SR median: 0.077  mean: 0.089  (right-skewed — most companies near-zero)
AVVR std: 0.292  (high spread — language style varies widely)
FRE mean: 27.1  (below 30 = 'Very Difficult' — BRSR text is hard to read)
BD: 9 of 15 companies score exactly 0.0 (most reports have no detectable boilerplate 

## 4. Bivariate Analysis — SR vs GRS

This is the core relationship of the project: does a report with more quantified claims (higher SR) show lower greenwashing risk (lower GRS)?  
We expect a negative correlation based on Clarkson et al. (2008) and Bingler et al. (2022).


In [ ]:
r, p = stats.pearsonr(df['SR'], df['GRS'])
print(f'Pearson r = {r:.4f}')
print(f'p-value   = {p:.4f}')
print(f'r²        = {r**2:.4f}  (SR explains {r**2*100:.1f}% of variance in GRS)')

# Regression line coefficients
m, b = np.polyfit(df['SR'], df['GRS'], 1)
print(f'\nRegression: GRS = {m:.1f} × SR + {b:.1f}')
print(f'Interpretation: for each 0.10 increase in SR (10 more quantified sentences per 100),'
      f' GRS drops by {abs(m)*0.1:.1f} points')

Pearson r = -0.7248
p-value   = 0.0022
r²        = 0.5253  (SR explains 52.5% of variance in GRS)

Regression: GRS = -111.6 × SR + 67.3
Interpretation: for each 0.10 increase in SR (10 more quantified sentences per 100), GRS drops by 11.2 points


## 5. 🔴 Surprise Finding — AVVR is a Stronger Predictor Than SR

**This was not expected at the project design stage.** We built GRS with SR carrying 40% weight (the dominant component) based on the literature.  
The data reveals that AVVR — Action-Vague Verb Ratio — actually has a *stronger* correlation with GRS than SR does.

> This is a real analytical finding: language *style* (whether verbs are action-oriented or aspirational) may be more diagnostic of greenwashing risk than the *presence* of numbers alone.

**What this means practically:** A report can include numbers (`SR > 0`) but still use them in aspirational constructions — *'we aim to achieve a 30% reduction'* scores a number (30%) but the verb *aims* counts as vague. AVVR catches this pattern; SR alone does not.


In [ ]:
r_sr,   p_sr   = stats.pearsonr(df['SR'],   df['GRS'])
r_avvr, p_avvr = stats.pearsonr(df['AVVR'], df['GRS'])
r_bd,   p_bd   = stats.pearsonr(df['BD'],   df['GRS'])
r_lm,   p_lm   = stats.pearsonr(df['LM_neg'], df['GRS'])

print('── Correlation of each GRS component with GRS ──\n')
results = [
    ('AVVR (Action-Vague Verb Ratio)', r_avvr, p_avvr, 25),
    ('SR   (Specificity Ratio)',        r_sr,   p_sr,   40),
    ('LM_neg (LM Finance Negative)',    r_lm,   p_lm,   15),
    ('BD   (Boilerplate Density)',       r_bd,   p_bd,   20),
]
for name, r, p, weight in sorted(results, key=lambda x: abs(x[1]), reverse=True):
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'
    print(f'{name:45s}  r={r:+.3f}  p={p:.4f} {sig}  (GRS weight: {weight}%)')

print('\n⚠  IMPLICATION FOR THE MODEL:')
print('AVVR has stronger empirical correlation with GRS than SR, yet SR carries higher weight.')
print('Future iterations should consider re-weighting: AVVR ≥ 40%, SR ≥ 25%.')
print('This is documented in the limitations section of the research report.')

── Correlation of each GRS component with GRS ──

AVVR (Action-Vague Verb Ratio)                  r=-0.802  p=0.0003 ***  (GRS weight: 25%)
SR   (Specificity Ratio)                        r=-0.725  p=0.0022 **   (GRS weight: 40%)
LM_neg (LM Finance Negative)                    r=+0.536  p=0.0394 *    (GRS weight: 15%)
BD   (Boilerplate Density)                      r=-0.041  p=0.8812 ns   (GRS weight: 20%)

⚠  IMPLICATION FOR THE MODEL:
AVVR has stronger empirical correlation with GRS than SR, yet SR carries higher weight.
Future iterations should consider re-weighting: AVVR ≥ 40%, SR ≥ 25%.
This is documented in the limitations section of the research report.


## 6. Sector Comparison

Do carbon-intensive sectors (Cement, Heavy Industry) really show higher greenwashing risk, as the international literature suggests?


In [ ]:
sector_stats = df.groupby('sector').agg(
    n=('GRS','count'),
    mean_GRS=('GRS','mean'),
    mean_SR=('SR','mean'),
    mean_AVVR=('AVVR','mean'),
    mean_FRE=('FRE','mean'),
    high_count=('GRS_tier', lambda x: (x=='HIGH').sum())
).round(3).sort_values('mean_GRS', ascending=False)

sector_stats['pct_HIGH'] = (sector_stats['high_count']/sector_stats['n']*100).round(0).astype(int)
print(sector_stats.to_string())

print('\n── Reading the table ──')
print('Heavy Industry and Cement have the highest mean GRS AND the lowest mean SR.')
print('This inverse pattern — high climate language, low quantitative grounding — matches')
print('Bingler et al. (2022): "cheap talk is most prevalent in carbon-intensive sectors."')
print('\nIT sector has the widest intra-sector spread (GRS range: 43.9–64.0, sd=10.6)')
print('Banking sector has a wide spread too (33.6–65.0, sd=16.3) — driven by HDFC outlier')

                n  mean_GRS  mean_SR  mean_AVVR  mean_FRE  high_count  pct_HIGH
sector
Heavy Industry  1     72.50    0.000      0.000     24.110           1       100
Cement          2     66.95    0.029      0.333     26.540           1        50
Pharma          3     66.10    0.086      0.417     31.427           2        67
IT              3     53.43    0.101      0.626     32.073           1        33
Banking         3     50.73    0.121      0.378     32.753           1        33
Auto            3     48.17    0.116      0.500     13.293           0         0

── Reading the table ──
Heavy Industry and Cement have the highest mean GRS AND the lowest mean SR.
This inverse pattern — high climate language, low quantitative grounding — matches
Bingler et al. (2022): "cheap talk is most prevalent in carbon-intensive sectors."

IT sector has the widest intra-sector spread (GRS range: 43.9–64.0, sd=10.6)
Banking sector has a wide spread too (33.6–65.0, sd=16.3) — driven by HDFC outlier

## 7. Outlier Analysis

Two companies score notably differently from their sector peers. Understanding *why* is important — these could be genuine outliers or pipeline artefacts.


In [ ]:
print('── ICICI Bank (lowest GRS = 33.6) ──')
icici = df[df.company=='ICICI Bank'].iloc[0]
print(f'  SR={icici.SR:.3f} (highest in corpus), AVVR={icici.AVVR:.3f} (highest), n_sentences={int(icici.n_sentences)}')
print(f'  Interpretation: 14 sentences, 4 quantified (SR=0.286), 4 action verbs (AVVR=0.800)')
print(f'  Short extract, data-dense. NOTE: could be artefact of extract brevity, not genuine best-in-corpus.')

print('\n── HDFC Bank (highest-GRS bank = 65.0) ──')
hdfc = df[df.company=='HDFC Bank'].iloc[0]
print(f'  SR={hdfc.SR:.3f}, AVVR={hdfc.AVVR:.3f}, n_sentences={int(hdfc.n_sentences)}')
print(f'  Oldest filing in corpus (FY2022-23). Zero quantified sentences in extract.')
print(f'  Possible explanation: BRSR quality improved significantly from FY23 to FY25')
print(f'  as SEBI introduced BRSR Core mandatory assurance requirements.')

print('\n── Infosys HIGH (GRS=64.0) despite strong actual ESG programme ──')
inf = df[df.company=='Infosys'].iloc[0]
print(f'  SR={inf.SR:.3f}, n_sentences={int(inf.n_sentences)} (largest extract in corpus)')
print(f'  Large extract includes substantial non-quantified narrative. Pipeline artefact:')
print(f'  longer documents dilute SR because governance sections and CEO letters are')
print(f'  prose-heavy. In a full-document analysis, Infosys SR would likely be higher.')

── ICICI Bank (lowest GRS = 33.6) ──
  SR=0.286 (highest in corpus), AVVR=0.800 (highest), n_sentences=14
  Interpretation: 14 sentences, 4 quantified (SR=0.286), 4 action verbs (AVVR=0.800)
  Short extract, data-dense. NOTE: could be artefact of extract brevity, not genuine best-in-corpus.

── HDFC Bank (highest-GRS bank = 65.0) ──
  SR=0.000, AVVR=0.000, n_sentences=14
  Oldest filing in corpus (FY2022-23). Zero quantified sentences in extract.
  Possible explanation: BRSR quality improved significantly from FY23 to FY25
  as SEBI introduced BRSR Core mandatory assurance requirements.

── Infosys HIGH (GRS=64.0) despite strong actual ESG programme ──
  SR=0.066, n_sentences=61 (largest extract in corpus)
  Large extract includes substantial non-quantified narrative. Pipeline artefact:
  longer documents dilute SR because governance sections and CEO letters are
  prose-heavy. In a full-document analysis, Infosys SR would likely be higher.


## 8. Readability Analysis

Flesch Reading Ease (FRE): 0–100. Higher = easier to read.  
- 60–70: Standard (plain English)  
- 30–50: Difficult (academic)  
- < 30: Very Difficult (legal/financial)

**Hypothesis:** If companies write BRSR reports to *satisfy regulators* rather than *inform investors*, they will use complex language that obscures rather than communicates.


In [ ]:
print('── Readability by sector (mean FRE) ──')
fre_sector = df.groupby('sector')['FRE'].mean().round(1).sort_values()
print(fre_sector.to_string())

print(f'\nCorpus mean FRE: {df["FRE"].mean():.1f} — "Very Difficult" threshold is <30')
print(f'Companies above 30 ("Difficult" or better): {(df["FRE"]>30).sum()} of 15')
print(f'Companies above 60 ("Standard"): {(df["FRE"]>60).sum()} of 15')

print('\n── Readability vs GRS ──')
r_fre, p_fre = stats.pearsonr(df['FRE'], df['GRS'])
print(f'r(FRE, GRS) = {r_fre:.3f}, p = {p_fre:.4f}')
print('Weak negative correlation: easier to read → slightly lower risk, but not significant.')

print('\n── Notable outliers ──')
print(df.nsmallest(3,'FRE')[['company','sector','FRE','GRS']].to_string(index=False))
print('M&M (FRE=0.20) and Maruti (FRE=4.24) have near-zero readability scores.')
print('These are Auto sector companies — complex technical language about multi-technology transitions.')

── Readability by sector (mean FRE) ──
sector
Auto              13.3
Heavy Industry    24.1
Cement            26.5
Pharma            31.4
IT                32.1
Banking           32.8
Name: FRE, dtype: float64

Corpus mean FRE: 27.1 — 'Very Difficult' threshold is <30
Companies above 30 ('Difficult' or better): 7 of 15
Companies above 60 ('Standard'): 0 of 15

── Readability vs GRS ──
r(FRE, GRS) = -0.307, p = 0.2668
Weak negative correlation: easier to read → slightly lower risk, but not significant.

── Notable outliers ──
  company sector   FRE   GRS
      M&M   Auto  0.20  49.3
   Maruti   Auto  4.24  46.3
UltraTech Cement 23.60  77.8
M&M (FRE=0.20) and Maruti (FRE=4.24) have near-zero readability scores.
These are Auto sector companies — complex technical language about multi-technology transitions.


## 9. Cross-Listing Effect (NYSE vs BSE/NSE-only)

**Hypothesis:** Companies cross-listed on NYSE face SEC reporting requirements and have more international institutional shareholders. Both incentivise higher disclosure specificity.

**Cross-listed companies in corpus:** Infosys (NYSE), Wipro (NYSE), Dr. Reddy's (NYSE) — 3 companies.


In [ ]:
df['nyse'] = df['exchange'].str.contains('NYSE')

nyse_df = df.groupby('nyse').agg(
    n=('GRS','count'),
    mean_SR=('SR','mean'),
    mean_AVVR=('AVVR','mean'),
    mean_GRS=('GRS','mean'),
    mean_FRE=('FRE','mean'),
).round(3)
nyse_df.index = ['BSE/NSE only','NYSE cross-listed']
print(nyse_df.to_string())

# T-test (note: n=3 vs n=12, very low power)
nyse_grs   = df[df.nyse==True]['GRS'].values
bse_grs    = df[df.nyse==False]['GRS'].values
t, p = stats.ttest_ind(nyse_grs, bse_grs)
print(f'\nT-test GRS (NYSE vs BSE/NSE): t={t:.3f}, p={p:.4f}')
print('Not significant — only 3 NYSE-listed companies. Direction is consistent with hypothesis.')
print(f'NYSE mean SR = {nyse_df.loc["NYSE cross-listed","mean_SR"]:.3f} vs BSE/NSE mean SR = {nyse_df.loc["BSE/NSE only","mean_SR"]:.3f}')

                  n  mean_SR  mean_AVVR  mean_GRS  mean_FRE
BSE/NSE only     12    0.079      0.389     58.408    25.488
NYSE cross-listed 3    0.126      0.654     53.767    29.720

T-test GRS (NYSE vs BSE/NSE): t=-0.721, p=0.4826
Not significant — only 3 NYSE-listed companies. Direction is consistent with hypothesis.
NYSE mean SR = 0.126 vs BSE/NSE mean SR = 0.079


## 10. Correlation Matrix

In [ ]:
corr_cols = ['SR','AVVR','BD','LM_pos','LM_neg','FRE','GRS']
corr = df[corr_cols].corr().round(3)
print('── Pearson Correlation Matrix ──\n')
print(corr.to_string())

print('\n── Key relationships ──')
print(f'SR–AVVR: r={corr.loc["SR","AVVR"]:.3f}  (expected — specific writers also use action verbs)')
print(f'SR–GRS:  r={corr.loc["SR","GRS"]:.3f}  (primary design relationship)')
print(f'AVVR–GRS: r={corr.loc["AVVR","GRS"]:.3f}  (**STRONGER** than SR–GRS)')
print(f'LM_neg–GRS: r={corr.loc["LM_neg","GRS"]:.3f}  (more negative words → higher risk)')
print(f'BD–GRS: r={corr.loc["BD","GRS"]:.3f}  (weakest signal — boilerplate density mostly zero)')

── Pearson Correlation Matrix ──

           SR   AVVR     BD  LM_pos  LM_neg     FRE    GRS
SR      1.000  0.606  0.084  -0.059  -0.115   0.247 -0.725
AVVR    0.606  1.000  0.148  -0.032  -0.012   0.240 -0.802
BD      0.084  0.148  1.000   0.093  -0.024   0.024 -0.041
LM_pos -0.059 -0.032  0.093   1.000   0.537  -0.028  0.112
LM_neg -0.115 -0.012 -0.024   0.537   1.000  -0.253  0.536
FRE     0.247  0.240  0.024  -0.028  -0.253   1.000 -0.307
GRS    -0.725 -0.802 -0.041   0.112   0.536  -0.307  1.000

── Key relationships ──
SR–AVVR: r=0.606  (expected — specific writers also use action verbs)
SR–GRS:  r=-0.725  (primary design relationship)
AVVR–GRS: r=-0.802  (**STRONGER** than SR–GRS)
LM_neg–GRS: r=0.536  (more negative words → higher risk)
BD–GRS: r=-0.041  (weakest signal — boilerplate density mostly zero)


## 11. Statistical Tests

Kruskal-Wallis: non-parametric test for differences across sectors. Chosen over ANOVA because:  
1. n per group is 1–3 (cannot verify normality)  
2. BRSR scores are bounded (0–100), not continuous normal


In [ ]:
from scipy.stats import kruskal, spearmanr

sectors = df['sector'].unique()
groups_grs = [df[df['sector']==s]['GRS'].values for s in sectors]
groups_sr  = [df[df['sector']==s]['SR'].values  for s in sectors]

h_grs, p_grs = kruskal(*groups_grs)
h_sr,  p_sr  = kruskal(*groups_sr)

print('── Kruskal-Wallis: sector differences ──')
print(f'GRS ~ sector:  H={h_grs:.3f}, p={p_grs:.4f}  {"*" if p_grs<0.05 else "ns"}')
print(f'SR  ~ sector:  H={h_sr:.3f},  p={p_sr:.4f}  {"*" if p_sr<0.05 else "ns"}')

print('\n── Spearman rank correlation (non-parametric) ──')
rho_sr,   p_rho_sr   = spearmanr(df['SR'],   df['GRS'])
rho_avvr, p_rho_avvr = spearmanr(df['AVVR'], df['GRS'])
print(f'ρ(SR, GRS)   = {rho_sr:.3f},   p = {p_rho_sr:.4f}')
print(f'ρ(AVVR, GRS) = {rho_avvr:.3f},   p = {p_rho_avvr:.4f}')

print('\n── Power analysis note ──')
print('With n=15 across 6 sectors (~2–3 per group), KW test has very low power.')
print('Failing to reject H0 does NOT confirm sector differences are absent.')
print('Required n for 80% power at α=0.05 to detect the observed effect size:')
print('Approximately 8–10 companies per sector (48–60 total BRSR reports).')

── Kruskal-Wallis: sector differences ──
GRS ~ sector:  H=7.467, p=0.1882  ns
SR  ~ sector:  H=4.990, p=0.4171  ns

── Spearman rank correlation (non-parametric) ──
ρ(SR, GRS)   = -0.726,   p = 0.0022
ρ(AVVR, GRS) = -0.737,   p = 0.0017

── Power analysis note ──
With n=15 across 6 sectors (~2–3 per group), KW test has very low power.
Failing to reject H0 does NOT confirm sector differences are absent.
Required n for 80% power at α=0.05 to detect the observed effect size:
Approximately 8–10 companies per sector (48–60 total BRSR reports).


## 12. Top 5 Actionable Findings

Summarising the EDA into findings a non-technical stakeholder can act on.


In [ ]:
findings = [
    ('FINDING 1', 'The LOW risk band is empty.',
     'None of the 15 reports scored GRS < 30. Even the best-performing report (ICICI Bank, GRS=33.6) is MEDIUM.'
     ' For investors: assume aspirational language in BRSR narratives until verified against KPI tables.'),

    ('FINDING 2', 'AVVR is the strongest single predictor of GRS (r=-0.802, p=0.0003).',
     'Language style — whether verbs are action-oriented or aspirational — is more diagnostic'
     ' than the presence of numbers alone. Screen for companies using past-tense action verbs'
     ' ("achieved", "reduced", "deployed") vs. future-tense aspirations ("aims", "seeks").'),

    ('FINDING 3', 'Manufacturing sectors show an inverse ambition-specificity pattern.',
     'Heavy Industry (GRS=72.5, SR=0.000) and Cement (GRS=66.95, SR=0.029) have the highest'
     ' climate language intensity (loaded on LDA Topic 4: Climate & GHG) but near-zero quantitative'
     ' grounding. Consistent with Bingler et al. (2022) on cheap talk in carbon-intensive sectors.'),

    ('FINDING 4', 'BRSR reports are uniformly difficult to read (mean FRE=27.1).',
     'Zero companies score above FRE=60 (Standard). Two Auto sector companies score near-zero readability.'
     ' If BRSR is intended to inform retail investors, the current language complexity defeats that purpose.'),

    ('FINDING 5', 'GRS weights need empirical recalibration in a future study.',
     'BD (Boilerplate Density) carries 20% weight in GRS but has near-zero empirical correlation'
     ' with GRS (r=-0.041) because most extract-based texts show BD=0. AVVR should carry higher'
     ' weight than SR based on empirical correlation. Recommending: AVVR 40%, SR 25%, BD 15%, LM_neg 20%.'),
]

for tag, title, body in findings:
    print(f'[{tag}] {title}')
    print(f'  → {body}')
    print()

[FINDING 1] The LOW risk band is empty.
  → None of the 15 reports scored GRS < 30. Even the best-performing report (ICICI Bank, GRS=33.6) is MEDIUM. For investors: assume aspirational language in BRSR narratives until verified against KPI tables.

[FINDING 2] AVVR is the strongest single predictor of GRS (r=-0.802, p=0.0003).
  → Language style — whether verbs are action-oriented or aspirational — is more diagnostic than the presence of numbers alone. Screen for companies using past-tense action verbs ("achieved", "reduced", "deployed") vs. future-tense aspirations ("aims", "seeks").

[FINDING 3] Manufacturing sectors show an inverse ambition-specificity pattern.
  → Heavy Industry (GRS=72.5, SR=0.000) and Cement (GRS=66.95, SR=0.029) have the highest climate language intensity (loaded on LDA Topic 4: Climate & GHG) but near-zero quantitative grounding. Consistent with Bingler et al. (2022) on cheap talk in carbon-intensive sectors.

[FINDING 4] BRSR reports are uniformly difficult to

## 13. Honest Limitations

*Every claim in this analysis should be read in light of these constraints.*

| Limitation | Severity | Impact |
|---|---|---|
| n=15 total, 1–3 per sector | High | Sector-level comparisons are descriptive only; Kruskal-Wallis has <20% power |
| Extract-based text for 8 of 15 reports | High | SR likely underestimated for reports with rich KPI tables later in document |
| GRS weights are theoretical, not fitted | Medium | Weights informed by literature but not validated against labelled data |
| FinBERT not integrated live | Medium | Transformer sentiment scores available only via Colab notebook extension |
| Lexicon coverage gaps (e.g., 'certified' missing from action verb list) | Medium | AVVR may undercount action verbs for certification-heavy reports (see Cipla) |
| Corpus spans 3 fiscal years | Low–Medium | BRSR Core introduced FY2023-24 changes disclosure requirements mid-sample |
| No performance cross-reference | High | Cannot detect actual greenwashing without matching CDP/SEBI emission data |

**The honest bottom line:** This pipeline measures *textual credibility risk*, not *actual greenwashing*.  
A HIGH score means: 'the language of this report makes independent verification difficult.'  
It does not mean: 'this company is definitely greenwashing.'

For actual greenwashing detection, the next step is cross-referencing these scores against  
CDP India disclosure data or MoEFCC emission filings — a future research extension.
